# Fine-tuning com Correções Reais — Classificador de Soja

Este notebook pega o modelo já treinado (`soja_model_final.keras`) e o **ajusta** com as fotos
que você corrigiu no app (salvas no dataset `Guguinhaxd/soja-correction`).

## Por que as correções têm peso máximo aqui

Uma correção manual não é dado qualquer — é sinal de qualidade máxima:
1. É uma foto do **seu** celular, no **seu** setup (resolve domain shift diretamente)
2. O modelo **já errou** nela → é exatamente onde ele precisa aprender
3. Você **viu o grão** e disse a classe certa → é ground truth confiável

Por isso as correções recebem **peso maior na loss**, dominam o mix de treino (70/30) e levam
**augmentation forte**. O dataset original (30%) entra só pra evitar *catastrophic forgetting*.

### Pré-requisitos
1. `soja_model_final.keras` no Drive (gerado pelo `train.ipynb`)
2. Correções no dataset HF (quanto mais por classe, melhor)
3. GPU ligada: `Ambiente de execução → Alterar tipo → T4 GPU`

### Ordem de execução
Rode as células **em ordem, sem pular** — cada uma depende da anterior.

In [ ]:
# Célula 1 — GPU, imports e login no Hugging Face
!pip install -q huggingface_hub

import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from huggingface_hub import snapshot_download, HfApi

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
assert len(gpus) > 0, 'Sem GPU! Ambiente de execução → Alterar tipo → T4'

# Pegue seu token em: https://huggingface.co/settings/tokens
# Precisa de permissão de leitura no dataset e escrita no Space.
from getpass import getpass
HF_TOKEN = getpass('Cole seu token do Hugging Face: ')

In [ ]:
# Célula 2 — Montar Drive, caminhos e classes
from google.colab import drive
drive.mount('/content/drive')

DATASET_ROOT     = '/content/drive/MyDrive/SoyaBeans Classifications.v2i.folder (Unzipped Files)'
TRAIN_DIR        = f'{DATASET_ROOT}/train'
TEST_DIR         = f'{DATASET_ROOT}/test'
SAVE_DIR         = '/content/drive/MyDrive'
BASE_MODEL_PATH  = f'{SAVE_DIR}/soja_model_final.keras'
CORRECTIONS_REPO = 'Guguinhaxd/soja-correction'

# Mesma ordem do train.ipynb — NÃO mudar (índice = saída do modelo)
CLASS_NAMES = [
    'Broken soybeans',
    'Immature soybeans',
    'Intact soybeans',
    'Skin-damaged soybeans',
    'Spotted soybeans',
]
SHORT = {
    'Broken soybeans':       'broken',
    'Immature soybeans':     'immature',
    'Intact soybeans':       'intact',
    'Skin-damaged soybeans': 'skin-damaged',
    'Spotted soybeans':      'spotted',
}
NUM_CLASSES = len(CLASS_NAMES)

assert os.path.isfile(BASE_MODEL_PATH), f'Modelo base não encontrado: {BASE_MODEL_PATH}'
assert os.path.isdir(TRAIN_DIR), f'Dataset original não encontrado: {TRAIN_DIR}'
print('OK — modelo base e dataset original encontrados.')

In [ ]:
# Célula 3 — Baixar correções do Hugging Face e contar por classe
CORR_DIR = snapshot_download(
    repo_id=CORRECTIONS_REPO,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/content/correcoes',
)
print(f'Correções baixadas em: {CORR_DIR}\n')

corr_counts = {}
for cls in CLASS_NAMES:
    files = (glob.glob(os.path.join(CORR_DIR, cls, '*.jpg'))  +
             glob.glob(os.path.join(CORR_DIR, cls, '*.jpeg')) +
             glob.glob(os.path.join(CORR_DIR, cls, '*.png')))
    corr_counts[cls] = files
    print(f'{SHORT[cls]:>14}: {len(files):>3} correções')

total_corr = sum(len(v) for v in corr_counts.values())
print(f'\nTotal de correções: {total_corr}')
assert total_corr >= 5, 'Poucas correções! Colete mais fotos pelo app antes de re-treinar.'
if total_corr < NUM_CLASSES * 8:
    print('⚠️  Recomendado ~8+ por classe. Dá pra rodar mesmo assim, mas quanto mais, melhor.')

In [ ]:
# Célula 4 — Recortar cada grão (MESMA lógica do app.py)
# Por que: o modelo classifica o grão RECORTADO (preenchendo o quadro), igual ao dataset
# original. As correções precisam ter o mesmo enquadramento, senão treino != inferência.
# As primeiras fotos foram salvas inteiras; esta célula normaliza TODAS.
# É idempotente: imagem já recortada (grão >95% do quadro) é mantida como está.
import cv2
from PIL import Image as PILImage

def crop_single_grain(arr):
    h, w    = arr.shape[:2]
    gray    = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
    thresh  = cv2.morphologyEx(thresh, cv2.MORPH_OPEN,  kernel, iterations=1)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return arr
    largest = max(contours, key=cv2.contourArea)
    if not (0.02 < cv2.contourArea(largest) / (h * w) < 0.95):
        return arr
    x, y, bw, bh = cv2.boundingRect(largest)
    pad = max(15, int(min(bw, bh) * 0.12))
    return arr[max(0, y-pad):min(h, y+bh+pad), max(0, x-pad):min(w, x+bw+pad)]

CROP_DIR = '/content/correcoes_cropped'
cropped_counts = {}
for cls in CLASS_NAMES:
    out_dir = os.path.join(CROP_DIR, cls)
    os.makedirs(out_dir, exist_ok=True)
    saved = []
    for fp in corr_counts[cls]:
        try:
            arr  = np.array(PILImage.open(fp).convert('RGB'))
            crop = crop_single_grain(arr)
            dst  = os.path.join(out_dir, os.path.basename(fp))
            PILImage.fromarray(crop).save(dst, format='JPEG', quality=95)
            saved.append(dst)
        except Exception as e:
            print(f'  pulei {os.path.basename(fp)}: {e}')
    cropped_counts[cls] = saved
    print(f'{SHORT[cls]:>14}: {len(saved):>3} recortadas')

corr_counts = cropped_counts
print('\nRecorte concluído — todas as correções no mesmo enquadramento do modelo.')

In [ ]:
# Célula 5 — Split treino/validação ESTRATIFICADO por classe
# Estratificado = separa 20% DE CADA CLASSE para validação, garantindo que
# todas as classes apareçam na validação mesmo com quantidades desiguais.
# (Split aleatório simples poderia deixar uma classe inteira de fora.)
rng = np.random.default_rng(42)
ct_paths, ct_labels, cv_paths, cv_labels = [], [], [], []

for idx, cls in enumerate(CLASS_NAMES):
    files = list(corr_counts[cls])
    if not files:
        print(f'{SHORT[cls]:>14}: SEM correções — classe ficará só com o dataset original')
        continue
    rng.shuffle(files)
    n_val = max(1, int(len(files) * 0.2))   # pelo menos 1 por classe na validação
    cv_paths  += files[:n_val]
    cv_labels += [idx] * n_val
    ct_paths  += files[n_val:]
    ct_labels += [idx] * (len(files) - n_val)
    print(f'{SHORT[cls]:>14}: {len(files)-n_val:>3} treino | {n_val:>2} validação')

ct_paths  = np.array(ct_paths)
ct_labels = np.array(ct_labels, dtype=np.int32)
cv_paths  = np.array(cv_paths)
cv_labels = np.array(cv_labels, dtype=np.int32)

assert len(ct_paths) > 0, 'Nenhuma correção para treino!'
print(f'\nTotal → treino: {len(ct_paths)} | validação: {len(cv_paths)}')

In [ ]:
# Célula 6 — Pipelines tf.data com peso nas correções
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# Peso automático: com poucos dados o peso alto extrai mais sinal; com muitos
# dados, peso alto só aumenta o risco de overfitting (visto na rodada 1 com 57 fotos).
n_total_corr = len(ct_paths)
CORR_WEIGHT  = 2.0 if n_total_corr < 150 else 1.5
ORIG_WEIGHT  = 1.0
print(f'Correções de treino: {n_total_corr}  →  CORR_WEIGHT={CORR_WEIGHT}')

# Augmentation FORTE nas correções (fotos reais do setup do usuário).
# value_range padrão do RandomBrightness é (0, 255) — compatível com nossas imagens float 0-255.
aug_strong = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.25),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.12, width_factor=0.12),
    layers.RandomBrightness(0.25),
    layers.RandomContrast(0.25),
], name='aug_strong')

# Augmentation LEVE no dataset original (entra só como replay anti-esquecimento).
aug_light = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomBrightness(0.05),
], name='aug_light')

def load_image(path, label):
    img   = tf.io.read_file(path)
    img   = tf.io.decode_image(img, channels=3, expand_animations=False)
    img   = tf.image.resize(img, IMG_SIZE)
    img   = tf.cast(img, tf.float32)
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

# Dataset original (replay)
orig_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
    batch_size=None, label_mode='categorical', shuffle=True, seed=42,
).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)

# Dataset de correções
corr_raw = (
    tf.data.Dataset.from_tensor_slices((ct_paths, ct_labels))
    .shuffle(len(ct_paths), seed=42, reshuffle_each_iteration=True)
    .map(load_image, num_parallel_calls=AUTOTUNE)
)

# Padrão .batch() → .map(aug) → .unbatch(): as camadas Keras de augmentation
# esperam tensores 4D (batch, H, W, C). O terceiro elemento é o sample_weight,
# que o Keras aplica automaticamente na loss durante o fit().
corr_stream = (
    corr_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_strong(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * CORR_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

orig_stream = (
    orig_raw.repeat()
    .batch(BATCH_SIZE)
    .map(lambda x, y: (aug_light(x, training=True), y,
                       tf.ones([tf.shape(x)[0]]) * ORIG_WEIGHT), num_parallel_calls=AUTOTUNE)
    .unbatch()
)

# Mix 70% correções / 30% original.
# Influência total das correções: 70% presença × peso na loss (~3-5× vs original).
mixed = (
    tf.data.Dataset.sample_from_datasets(
        [corr_stream, orig_stream], weights=[0.7, 0.3], seed=42
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# Validação: suas fotos, sem augmentation e sem peso (mede acurácia real no seu domínio)
corr_val = (
    tf.data.Dataset.from_tensor_slices((cv_paths, cv_labels))
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ~8 exposições por correção por época (cada vez com augmentation diferente)
STEPS_PER_EPOCH = max(60, (len(ct_paths) * 8) // int(BATCH_SIZE * 0.7))
print(f'Steps por época: {STEPS_PER_EPOCH}'
      f'  (~{STEPS_PER_EPOCH * int(BATCH_SIZE * 0.7) / len(ct_paths):.0f}× por correção)')

In [ ]:
# Célula 7 — Carregar modelo, medir baseline e descongelar camadas
model = keras.models.load_model(BASE_MODEL_PATH)
print('Modelo carregado.')

# Acurácia ANTES do fine-tuning nas suas fotos — o baseline a superar
model.compile(loss='categorical_crossentropy', metrics=['accuracy'])
base_loss, base_acc = model.evaluate(corr_val, verbose=0)
print(f'\nACURÁCIA ANTES (nas suas fotos): {base_acc:.1%}')

# Descongela as últimas 30 camadas da base EfficientNet.
# BatchNorm fica congelada: descongelar destrói as estatísticas (mean/variance)
# herdadas do ImageNet e piora o resultado.
base_model = next((l for l in model.layers if isinstance(l, keras.Model)), None)
if base_model is not None:
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    trainable = sum(1 for l in base_model.layers if l.trainable)
    print(f'Camadas treináveis na base: {trainable}/{len(base_model.layers)}')
else:
    print('Base aninhada não encontrada — treinando só a cabeça (ainda funciona).')

In [ ]:
# Célula 8 — Fine-tuning
# LR 100× menor que o treino original: ajusta sem destruir os pesos aprendidos
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=6, restore_best_weights=True,
        monitor='val_accuracy', verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{SAVE_DIR}/soja_finetuned_best.keras',
        save_best_only=True, monitor='val_accuracy', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        patience=3, factor=0.5, monitor='val_loss',
        min_lr=1e-7, verbose=1),
]

print('=== Fine-tuning — 70% correções / 30% original ===')
print(f'CORR_WEIGHT={CORR_WEIGHT} | epochs=20 | steps/epoch={STEPS_PER_EPOCH}')
history = model.fit(
    mixed,
    validation_data=corr_val,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks,
)

In [ ]:
# Célula 9 — Comparar ANTES vs DEPOIS + matriz de confusão por classe
new_loss, new_acc = model.evaluate(corr_val, verbose=0)
print(f'ANTES  — acurácia nas suas fotos: {base_acc:.1%}')
print(f'DEPOIS — acurácia nas suas fotos: {new_acc:.1%}')
print(f'Ganho: {(new_acc - base_acc)*100:+.1f} pontos percentuais')

# Sanidade: o modelo NÃO pode ter esquecido o dataset original
test_ds = (
    tf.keras.utils.image_dataset_from_directory(
        TEST_DIR, class_names=CLASS_NAMES, image_size=IMG_SIZE,
        batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False,
    )
    .map(lambda x, y: (tf.cast(x, tf.float32), y))
    .prefetch(AUTOTUNE)
)
_, test_acc = model.evaluate(test_ds, verbose=0)
print(f'\nAcurácia no TESTE original: {test_acc:.1%}  (não pode despencar abaixo de ~70%)')

# Matriz de confusão nas SUAS fotos — mostra exatamente qual classe confunde com qual
y_pred = np.argmax(model.predict(corr_val, verbose=0), axis=1)
y_true = cv_labels
short_names = [SHORT[c] for c in CLASS_NAMES]

from sklearn.metrics import confusion_matrix, classification_report
print('\n=== Relatório por classe (suas fotos) ===')
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES),
                            target_names=short_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot(history.history['accuracy'],     label='Treino (misto)')
axes[0].plot(history.history['val_accuracy'], label='Validação (suas fotos)')
axes[0].set_title('Acurácia'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Treino')
axes[1].plot(history.history['val_loss'], label='Validação')
axes[1].set_title('Loss'); axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(alpha=0.3)

im = axes[2].imshow(cm, cmap='Blues')
axes[2].set_xticks(range(NUM_CLASSES)); axes[2].set_xticklabels(short_names, rotation=45, ha='right')
axes[2].set_yticks(range(NUM_CLASSES)); axes[2].set_yticklabels(short_names)
axes[2].set_xlabel('Predito'); axes[2].set_ylabel('Real')
axes[2].set_title('Matriz de confusão (suas fotos)')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        axes[2].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/soja_finetune_resultado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráficos salvos no Drive: soja_finetune_resultado.png')

In [ ]:
# Célula 10 — Salvar
# Só sobrescreve o modelo de produção se melhorou nas suas fotos
# E não destruiu o desempenho no dataset original.
FINETUNED_PATH = f'{SAVE_DIR}/soja_finetuned_final.keras'
model.save(FINETUNED_PATH)
print(f'Modelo ajustado salvo em: {FINETUNED_PATH}')

if new_acc >= base_acc and test_acc >= 0.70:
    model.save(f'{SAVE_DIR}/soja_model_final.keras')
    print('\n✅ Melhorou nas suas fotos e segurou o teste original.')
    print('   Sobrescrevi soja_model_final.keras — pronto para o Space.')
else:
    print('\n⚠️  Resultado abaixo do esperado. Mantive o soja_model_final.keras antigo.')
    print('   Colete mais correções (veja na matriz de confusão QUAL classe precisa de mais fotos).')
    if test_acc < 0.70:
        print('   O modelo esqueceu demais o dataset original — reduza CORR_WEIGHT para 1.0-1.5')
        print('   ou aumente a fração do original no mix (ex.: 60/40) e rode de novo.')

In [ ]:
# Célula 11 (OPCIONAL) — Subir o modelo direto pro HF Space via API
# Evita o download/git manual do .keras (29 MB com LFS).
# Só rode se a Célula 10 mostrou ✅.
SPACE_REPO = 'Guguinhaxd/soja-inspection'

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=f'{SAVE_DIR}/soja_model_final.keras',
    path_in_repo='soja_model_final.keras',
    repo_id=SPACE_REPO,
    repo_type='space',
    commit_message='fine-tune: modelo ajustado com correções reais',
)
print('✅ Modelo enviado pro Space. Ele vai rebuildar sozinho em ~2 min.')